In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse

import pandas as pd
from torch.autograd import Variable
import importlib.util
from torch.utils.data import DataLoader
from datasets import Dataset
module_path = r"D:/timeseries/package/state-spaces-simple/src/models/sequence/ss/standalone/s4.py"

spec = importlib.util.spec_from_file_location("S4", module_path)
S4 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(S4)
from torch.optim import AdamW

from sqlalchemy import create_engine
from sqlalchemy import text

CUDA extension for cauchy multiplication not found. Install by going to extensions/cauchy/ and running `python setup.py install`. This should speed up end-to-end training by 10-50%
Falling back on slow Cauchy kernel. Install at least one of pykeops or the CUDA extension for efficiency.


In [26]:
data=pd.read_csv('D:/timeseries/data/data_5.csv')


In [15]:
data

,DATE_TIME,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST,ERCOT
0,2002-01-01 01:00:00,8331.469266,1111.096549,1094.045496,995.298392,10336.304899,2165.007571,4793.193560,843.747177,29670.162911
1,2002-01-01 02:00:00,8107.805431,1091.481584,1083.517981,981.195477,10178.052738,2092.374118,4766.918187,835.488188,29136.833703
2,2002-01-01 03:00:00,7890.721717,1080.257616,1085.038740,971.305257,10100.887710,2042.080714,4702.918892,830.694878,28703.905525
3,2002-01-01 04:00:00,7799.817527,1079.726403,1093.381853,971.262963,10081.565109,2011.935791,4669.064391,835.068413,28541.822450
4,2002-01-01 05:00:00,7815.968171,1087.934961,1106.651296,982.535591,10192.218670,2002.487678,4695.011904,848.598209,28731.406478
...,...,...,...,...,...,...,...,...,...,...
187698,2023-05-31 20:00:00,16830.963445,2173.074464,5757.713701,1430.314138,18926.254677,5160.743439,10820.919849,1471.589427,62571.573139
187699,2023-05-31 21:00:00,16249.277372,2061.613777,5712.504436,1357.031324,18042.293868,4963.232330,10363.331700,1412.062420,60161.347225
187700,2023-05-31 22:00:00,15690.099605,1953.621682,5696.530667,1305.911891,17295.147115,4810.546445,9939.491289,1384.384586,58075.733278
187701,2023-05-31 23:00:00,14800.724443,1802.798146,5605.837507,1224.213988,16001.147539,4521.148141,9248.031281,1305.815735,54509.716780


In [14]:
names=list(set(data["SYMBOL"]))

KeyError: 'SYMBOL'

In [3]:
def transform_data(sel,names):
  
  datamain=sel.query("SYMBOL=='%s'"%(names[0]))
  datamain=datamain[["DATE","PRICE"]]
  datamain.drop_duplicates(subset=["DATE"], keep='first', inplace=True)
  datamain=datamain.sort_values(by="DATE")
  s=names[0]+"_price"
  s2=names[0]+"_size"
  # datamain["DATE"]=pd.to_datetime(datamain["DATE"])
  datamain=datamain.rename(columns={"PRICE":s})
  # datamain[s] = pd.to_numeric(datamain["PRICE"])
  # datamain[s2] = pd.to_numeric(datamain["SIZE"])


  for name in names:
    if name==names[0]:
      continue
    data=sel.query("SYMBOL=='%s'"%name)

    data=data[["DATE","PRICE"]]
    data.drop_duplicates(subset=["DATE"], keep='first', inplace=True)
    
    # data["PRICE"]=pd.to_datetime(data["PRICE"])
    # data["Close_"] = pd.to_numeric(data["Close_"])
 
    data=data.rename(columns={"PRICE": "%s_price"%name})


    datamain=pd.merge(datamain,data,on="DATE", how="outer")
  return datamain

In [92]:
new_data=transform_data(data,names)

In [4]:
def normalize(data):
    data=data.astype(float)
    mean_list=[]
    std_list=[]
    for i in data.columns:
        try:
          # mean=data[i].mean()
          mean=0
        except:
          print(data[i],i)
          break
        std=1
        std=data[i].std()
        data[i]=(data[i]-mean)/std
        # for j in range(len(data[i])):
          # if data[i][j]!=0:
          #   first=data[i][j]
          #   data[i]=data[i]/first+5
          #   break

        
        mean_list.append(mean)
        std_list.append(std)   
    # return data,first
    return data,mean_list,std_list

def get_mask(data):
    """
    data should in the form of pd.df
    gen a tenor with 0 and 1 to represent missing data
    """
    mask = ~data.isnan().values
 
    mask_tensor = torch.tensor(mask, dtype=torch.float32)
    
    mask_tensor= mask_tensor.transpose(0,1)
    return mask_tensor

def mape(A,F,maskf_sub):
  sum=0
  length=0
  for i in range(len(A)):
   
    if maskf_sub[i]!=0:
      sum+=abs(A[i] - F[i]) / abs(A[i])
      length+=1
  if length>0:
    return 100/length*sum
  
  return 0
def smape(A, F,maskf_sub):
  sum=0
  length=0
  for i in range(len(A)):
    if maskf_sub[i]!=0:
      sum+=2 * abs(F[i] - A[i]) / (abs(A[i]) + abs(F[i]))
      length+=1
  if length>0:

    return 100/length * sum
  return 0

In [5]:

def create_inputs(data, context_length, prediction_length):
    num_days, num_products = data.shape
    num_samples = num_days - context_length - prediction_length + 1
   

    samples = torch.zeros((num_samples, context_length,num_products))

    for i in range(num_samples):
        samples[i,:,:] = data[i:i+context_length]

    return samples,num_samples
def create_targets(data,context_length,prediction_length):
    num_days, num_products = data.shape
    num_samples = num_days - context_length - prediction_length + 1
    

    targets = torch.zeros((num_samples, prediction_length,num_products))

    for i in range(num_samples):
        targets[i, :,:] = data[i+context_length:i+prediction_length+context_length]

    return targets




def split_train_val(data,prediction_period,batchs,context_length,col_len):
    date=data.iloc[:,1]
    whole,m,std=normalize(data.iloc[:,1:])
    # whole,m,std=normalize(data)
    whole=torch.tensor(whole.values)
    # whole=whole.transpose(0,1)
   
    inputs,period=create_inputs(whole,context_length,prediction_period)
    target=create_targets(whole,context_length,prediction_period)
    inputs=inputs.reshape(period,1,col_len*context_length)
    target=target.reshape(period,1,prediction_period*col_len)
    # print(inputs.shape)
    # train_input=inputs[:period-prediction_period].transpose(0,1)
    # test_input=inputs[:period-prediction_period].transpose(0,1)
    # train_target=target[:period-prediction_period].transpose(0,1)
    # test_target=target[period-prediction_period:].transpose(0,1)
    train_input=inputs[:period-prediction_period]
    test_input=inputs[-1:]
    train_target=target[:period-prediction_period]
    test_target=target[-1:]
    print(test_input.shape,test_target.shape)
    
    traindict={'target':train_target,'input':train_input}
    testdict={'target':test_target,'input':test_input}
    train=Dataset.from_dict(traindict)
    train=train.with_format('torch')
    test=Dataset.from_dict(testdict)
    test=test.with_format('torch')
    

    train_loader = DataLoader(train, batch_size=batchs, shuffle=False)
    test_loader = DataLoader(test, batch_size=batchs, shuffle=False)
    return train_loader, test_loader,m,std,date

class S4Model(nn.Module):

    def __init__(
        self, 
        d_input, 
        d_output=10, 
        d_model=256, 
        n_layers=4, 
        dropout=0.2,
        prenorm=False,
    ):
        super().__init__()

        self.prenorm = prenorm

        # Linear encoder (d_input = 1 for grayscale and 3 for RGB)
        self.encoder = nn.Linear(d_input, d_model)

        # Stack S4 layers as residual blocks
        self.s4_layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(n_layers):
            self.s4_layers.append(
                S4.S4(
                    d_model=d_model, 
                    l_max=1024, 
                    bidirectional=True,
                    postact='glu',
                    dropout=dropout, 
                    transposed=True,
                )
            )
            self.norms.append(nn.LayerNorm(d_model))
            self.dropouts.append(nn.Dropout2d(dropout))

        # Linear decoder
        self.decoder = nn.Linear(d_model, d_output)

    def forward(self, x):
        """
        Input x is shape (B, L, d_input)
        """
        x = self.encoder(x)  # (B, L, d_input) -> (B, L, d_model)
        
        x = x.transpose(-1, -2)  # (B, L, d_model) -> (B, d_model, L)
        for layer, norm, dropout in zip(self.s4_layers, self.norms, self.dropouts):
            # Each iteration of this loop will map (B, d_model, L) -> (B, d_model, L)

            z = x
            if self.prenorm:
                # Prenorm
                z = norm(z.transpose(-1, -2)).transpose(-1, -2)
            
            # Apply S4 block: we ignore the state input and output
            z, _ = layer(z)

            # Dropout on the output of the S4 block
            z = dropout(z)

            # Residual connection
            x = z + x

            if not self.prenorm:
                # Postnorm
                x = norm(x.transpose(-1, -2)).transpose(-1, -2)

        x = x.transpose(-1, -2)

        # Pooling: average pooling over the sequence length
        x = x.mean(dim=1)

        # Decode the outputs
        x = self.decoder(x)  # (B, d_model) -> (B, d_output)

        return x

In [6]:
def train_model(epoch,train,ct,pt,col_len):
    model = S4Model(
    d_input=col_len*ct, 
    d_output=pt*col_len, 
    d_model=64, 
    n_layers=4, 

    dropout=0.1,
    prenorm=False
    )
    device='cpu'
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=0.001,)

    model.train()
    for epoch in range(epoch):
        for ind,batch in enumerate(train):
            optimizer.zero_grad()
            target=batch['target']
            inputs=batch['input']
            # target_mask = ~torch.isnan(target)
            # target_mask = target_mask.view(target.shape[0],target.shape[1],target.shape[2])
            # valid_target = target[target_mask]
            # input_mask=target_mask.reshape()
            # valid_inputs = inputs[target_mask]
        
            
            # inputs=Variable(inputs,requires_grad=True)
            # input_mask = ~torch.isnan(inputs)
            # input_mask=input_mask.reshape(inputs.shape[0],inputs.shape[1],inputs.shape[2])
            # print(input_mask.shape)
            # valid_inputs = inputs[input_mask]
            # print(valid_inputs.shape)
            
            # inputs=inputs.nan_to_num()
            inputs=Variable(inputs,requires_grad=True)
            
            outputs = model(inputs)
            # print(outputs)
            # output_mask=target_mask.reshape(outputs.shape[0],outputs.shape[1])
            criterion = nn.MSELoss()
            # valid_outputs=outputs[output_mask]
            # print(valid_outputs)
            # break
            loss =criterion(outputs,target)
            # loss =criterion(valid_outputs,valid_target)

            loss.backward()
            optimizer.step()
        
        if epoch % 20 == 0:
            print(loss.item())

        model.eval()
    return model

In [7]:
def predict(model,test,pt,col_len):
    pred=[]
    for ind,batch in enumerate(test):
        inputs=batch['input']
        inputs=inputs.nan_to_num()
        print(inputs,inputs.shape)
        out=model(inputs)
        out=out.reshape(pt,col_len)
        out=out.T
        pred=pred+list(out)
        
    return pred
            
def denormalize(data,mean,std):
    for i in range(len(data)):
        print(data[i],std[i],mean[i])
        data[i]=data[i]*std[i]+mean[i]
    return data       


In [8]:

def mape(A,F,maskf_sub):
  sum=0
  length=0
  for i in range(len(A)):
    # print(A[i],"A",maskf_sub[i],"ma")
    if not torch.isnan(A[i]):
    # if maskf_sub[i]!=0:
      sum+=abs(A[i] - F[i]) / abs(A[i])
      length+=1
  if length>0:
    return 100/length*sum
  
  return 0
def smape(A, F,maskf_sub):
  sum=0
  length=0
  for i in range(len(A)):
    if not torch.isnan(A[i]):
    # if maskf_sub[i]!=0:
      sum+=2 * abs(F[i] - A[i]) / (abs(A[i]) + abs(F[i]))
      length+=1
  if length>0:

    return 100/length * sum
  return 0

In [23]:
def cal_metrics(get,col_len,pred_len):
    smpl=[]
    mpl=[]
    m=get[4]
    std=get[5]
    act=next(iter(get[3]))
    act=(act['target']).reshape(pred_len,col_len)
    act=act.T
    deno_pred=get[2]
    act=denormalize(act,m,std)
    act_mask=(~torch.isnan(act))
    act_mask=act_mask.reshape(col_len*pred_len)

    # for i in range(len(deno_pred)):
    #     for j in range(len(deno_pred[i])):
    #         deno_pred[i][j]=float(deno_pred[i][j])
    new_pred=[]
    for i in range(col_len):
        new_pred.append([])
        for j in range(pred_len):
            new_pred[i].append(float(deno_pred[i][j]))

    for i in range(col_len):

        mp=mape(act[i],deno_pred[i],act_mask)
        smp=smape(act[i],deno_pred[i],act_mask)
        
        smpl.append(float(smp))
        mpl.append(float(mp))
    
    
    
    return mpl,smpl,new_pred,act

In [10]:
def S4_run(data,pred_length,batch_size,context_l,epoch,col_len):
    train,test,m,std,date=split_train_val(data,pred_length,batch_size,context_l,col_len)
    model=train_model(epoch,train,context_l,pred_length,col_len)
    pred=predict(model,test,pred_length,col_len)
    denormalize_pred=denormalize(pred,m,std)
    return model,pred,denormalize_pred,test,m,std,date
    

    

In [11]:
def S4_forward(data,pred_length,batch_size,context_l,epoch,col_len,step):
    next_data=data[:step]
    pace=(len(data)-step)//step
    month_mp=[]
    month_smp=[]
    pred=[]
    act=[]
    for i in range(col_len):
        month_mp.append([])
        month_smp.append([])
        pred.append([])
        act.append([])
    for i in range(1,30+1):
        get=S4_run(next_data,pred_length,batch_size,context_l,epoch,col_len)
        
        mpl,smpl,new_pred,acts=cal_metrics(get,col_len,pred_length)
        
        for q in range(col_len):
            month_mp[q].append(mpl[q])
            month_smp[q].append(smpl[q])
        

            pred[q]=pred[q]+list(new_pred[q])
            act[q]=act[q]+list(acts[q])

        next_data=data[step*pace:step+step*pace]
    return month_mp,month_smp,pred,act




In [27]:

month_mp,month_smp,pred,act=S4_forward(data,3,30,6,50,15,30)


torch.Size([1, 1, 90]) torch.Size([1, 1, 45])


d:\Users\lib\site-packages\torch\nn\functional.py:1340: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "
d:\Users\lib\site-packages\torch\nn\modules\loss.py:536: UserWarning: Using a target size (torch.Size([19, 1, 45])) that is different to the input size (torch.Size([19, 45])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


1796.374267578125
1687.5643310546875
1607.218994140625
tensor([[[13.3712, 28.8884, 76.7526, 14.0105, 29.6106, 77.9202, 13.5194,
          29.2890, 77.1330, 13.9521, 27.5797, 75.2451,  1.5891,  4.6911,
           5.6250, 13.0658, 29.5754, 75.9403, 13.9273, 29.1417, 77.6262,
          13.4575, 29.5979, 76.6050, 14.1573, 26.3565, 75.1027,  1.1398,
           1.8576,  4.4349, 13.2384, 28.6637, 76.3464, 13.8395, 28.3746,
          77.5378, 13.4919, 29.1365, 76.7220, 13.9380, 26.4181, 74.5331,
           0.8780,  2.1195,  4.3428, 13.2550, 27.4951, 75.8824, 13.9123,
          28.1898, 77.3321, 13.4873, 28.1168, 76.4875, 14.1279, 26.2630,
          74.8179,  0.8170,  1.8537,  4.0958, 13.5847, 27.8618, 76.1142,
          14.2681, 27.3184, 77.4494, 13.7109, 27.9760, 76.7810, 14.1526,
          25.2011, 74.5899,  1.3865,  1.6419,  4.3497, 13.2218, 26.8712,
          75.6213, 14.3755, 27.5345, 77.3615, 13.5917, 27.2467, 76.4286,
          14.5052, 25.6876, 74.7611,  1.8561,  2.4560,  5.5219]]]) to

In [29]:
names=data.columns[1:]

In [30]:
result= pd.DataFrame({'lstm_mape_%s'%names[0]:month_mp[0],'lstm_smape_%s'%names[0]:month_smp[0]
                      ,'lstm_mape_%s'%names[1]:month_mp[1],'lstm_smape_%s'%names[1]:month_smp[1],
                      'lstm_mape_%s'%names[2]:month_mp[2],'lstm_smape_%s'%names[2]:month_smp[2],
                      'lstm_mape_%s'%names[3]:month_mp[3],'lstm_smape_%s'%names[3]:month_smp[3],
                      'lstm_mape_%s'%names[4]:month_mp[4],'lstm_smape_%s'%names[4]:month_smp[4],
                      'lstm_mape_%s'%names[5]:month_mp[5],'lstm_smape_%s'%names[5]:month_smp[5],
                      'lstm_mape_%s'%names[6]:month_mp[6],'lstm_smape_%s'%names[6]:month_smp[6],
                      'lstm_mape_%s'%names[7]:month_mp[7],'lstm_smape_%s'%names[7]:month_smp[7],
                      'lstm_mape_%s'%names[8]:month_mp[8],'lstm_smape_%s'%names[8]:month_smp[8],
                      'lstm_mape_%s'%names[9]:month_mp[9],'lstm_smape_%s'%names[9]:month_smp[9],
                      'lstm_mape_%s'%names[10]:month_mp[10],'lstm_smape_%s'%names[10]:month_smp[10],
                      'lstm_mape_%s'%names[11]:month_mp[11],'lstm_smape_%s'%names[11]:month_smp[11],
                      'lstm_mape_%s'%names[12]:month_mp[12],'lstm_smape_%s'%names[12]:month_smp[12],
                      'lstm_mape_%s'%names[13]:month_mp[13],'lstm_smape_%s'%names[13]:month_smp[13],
                      'lstm_mape_%s'%names[14]:month_mp[14],'lstm_smape_%s'%names[14]:month_smp[14],
                     #  'lstm_mape_%s'%names[15]:month_mp[15],'lstm_smape_%s'%names[15]:month_smp[15]
                      })
result.to_csv('D:/timeseries/result/temp_2.csv')
res2= pd.DataFrame({'pred_%s'%names[0]:pred[0], 'actu_%s'%names[0]:act[0],
                    'pred_%s'%names[1]:pred[1], 'actu_%s'%names[1]:act[1],
                    'pred_%s'%names[2]:pred[2], 'actu_%s'%names[2]:act[2],
                    'pred_%s'%names[3]:pred[3], 'actu_%s'%names[3]:act[3],
                    'pred_%s'%names[4]:pred[4], 'actu_%s'%names[4]:act[4],
                    'pred_%s'%names[5]:pred[5], 'actu_%s'%names[5]:act[5],
                    'pred_%s'%names[6]:pred[6], 'actu_%s'%names[6]:act[6],
                    'pred_%s'%names[7]:pred[7], 'actu_%s'%names[7]:act[7],
                    'pred_%s'%names[8]:pred[8], 'actu_%s'%names[8]:act[8],
                    'pred_%s'%names[9]:pred[9], 'actu_%s'%names[9]:act[9],
                    'pred_%s'%names[10]:pred[10], 'actu_%s'%names[10]:act[10],
                    'pred_%s'%names[11]:pred[11], 'actu_%s'%names[11]:act[11],
                    'pred_%s'%names[12]:pred[12], 'actu_%s'%names[12]:act[12],
                    'pred_%s'%names[13]:pred[13], 'actu_%s'%names[13]:act[13],
                    'pred_%s'%names[14]:pred[14], 'actu_%s'%names[14]:act[14],
                    
                    
                    
                       })
res2.to_csv('D:/timeseries/result/temp.csv')

In [ ]:
data=pd.read_csv('D:/timeseries/data/data_4.csv')

In [ ]:
month_mp,month_smp,pred,act=S4_forward(data,3,30,6,50,15,30)

In [138]:

result= pd.DataFrame({'S4_mape':mpl,'S4_smape':smpl})
result.to_csv('D:/timeseries/result/temp_2.csv')
res2= pd.DataFrame({'name':names,'day1/1_pred':new_pred[0], 'day1/1_actu':act[0]
                       })
res2.to_csv('D:/timeseries/result/temp.csv')

PermissionError: [Errno 13] Permission denied: 'D:/timeseries/result/temp.csv'

In [119]:
# act=act.T
result= pd.DataFrame({'S4_mape':mpl,'S4_smape':smpl})
result.to_csv('D:/timeseries/result/temp_2.csv')
res2= pd.DataFrame({'name':names,'day1/7_pred':act[0], 'day1/7_actu':new_pred[0],'day2/7_pred':act[1], 
                    'day2/7_actu':new_pred[1],'day3/7_pred':act[2], 'day3/7_actu':new_pred[2],'day4/7_pred':act[3], 'day4/7_actu':new_pred[3]
                    ,'day5/7_pred':act[4], 'day5/7_actu':new_pred[4],'day6/7_pred':act[5], 'day6/7_actu':new_pred[5],
                    'day7/7_pred':act[6], 'day7/7_actu':new_pred[6]
                       })
res2.to_csv('D:/timeseries/result/temp.csv')